# Task 3.2 - Failure Mode

**Failure scenario:** A nearly constant time series where many subsequences are extremely similar after normalization. I expected the method to struggle because lower bounds become very loose, so the pruning cascade would fail and DTW would be computed for almost every candidate.

In [1]:

import numpy as np
import time

data = np.load("partB/data/failure_series.npz")
T = data["T"]
Q = data["Q"]

m = len(Q)
r = max(1, int(0.05 * m))

def z_normalize(ts):
    mean = ts.mean()
    std = ts.std()
    if std < 1e-8:
        return ts - mean
    return (ts - mean) / std

def envelope(ts, r):
    n = len(ts)
    upper = np.empty(n)
    lower = np.empty(n)
    for i in range(n):
        start = max(0, i - r)
        end = min(n, i + r + 1)
        window = ts[start:end]
        upper[i] = window.max()
        lower[i] = window.min()
    return upper, lower

def lb_kim_fl(q, c):
    return (q[0] - c[0]) ** 2 + (q[-1] - c[-1]) ** 2

def lb_keogh(c, upper, lower, bsf):
    s = 0.0
    for i, v in enumerate(c):
        if v > upper[i]:
            d = (v - upper[i]) ** 2
        elif v < lower[i]:
            d = (v - lower[i]) ** 2
        else:
            d = 0.0
        s += d
        if s > bsf:
            return s
    return s

def dtw_distance(q, c, r, bsf):
    m = len(q)
    w = r
    INF = 1e18
    dtw = np.full((m + 1, m + 1), INF)
    dtw[0, 0] = 0.0
    for i in range(1, m + 1):
        start = max(1, i - w)
        end = min(m, i + w)
        row_min = INF
        for j in range(start, end + 1):
            cost = (q[i - 1] - c[j - 1]) ** 2
            dtw[i, j] = cost + min(dtw[i - 1, j], dtw[i, j - 1], dtw[i - 1, j - 1])
            row_min = min(row_min, dtw[i, j])
        if row_min > bsf:
            return row_min
    return dtw[m, m]

def ucr_dtw_search(T, Q, r):
    q = z_normalize(Q)
    U, L = envelope(q, r)
    bsf = float('inf')
    for i in range(0, len(T) - len(Q) + 1):
        c = z_normalize(T[i:i + len(Q)])
        if lb_kim_fl(q, c) >= bsf:
            continue
        if lb_keogh(c, U, L, bsf) >= bsf:
            continue
        Uc, Lc = envelope(c, r)
        if lb_keogh(q, Uc, Lc, bsf) >= bsf:
            continue
        d = dtw_distance(q, c, r, bsf)
        if d < bsf:
            bsf = d
    return bsf

def naive_search(T, Q, r):
    q = z_normalize(Q)
    bsf = float('inf')
    for i in range(0, len(T) - len(Q) + 1):
        c = z_normalize(T[i:i + len(Q)])
        d = dtw_distance(q, c, r, bsf)
        if d < bsf:
            bsf = d
    return bsf

def timed(fn, runs=3):
    times = []
    last = None
    for _ in range(runs):
        start = time.perf_counter()
        last = fn()
        times.append(time.perf_counter() - start)
    return float(np.mean(times)), last

naive_time, _ = timed(lambda: naive_search(T, Q, r))
ucr_time, _ = timed(lambda: ucr_dtw_search(T, Q, r))

print(f"Naive DTW time: {naive_time:.4f}s")
print(f"UCR-DTW time:  {ucr_time:.4f}s")


Naive DTW time: 2.3912s
UCR-DTW time:  3.1907s


**Interpretation:** In this dataset the series is nearly constant, so Z-normalization yields sequences with very small variation. The envelopes are narrow and many candidates fall inside them, making LB_Keogh close to zero. As a result, the cascade prunes almost nothing and DTW is computed for nearly every subsequence. The extra bound computations add overhead, so UCR-DTW is slower than the naive baseline here. This failure is directly tied to the assumption that lower bounds are tight on real data; when that fails, the method's advantage disappears.

**Suggested modification:** Add an adaptive switch that skips lower-bound computation when a quick variance test indicates the data are nearly constant.

In [1]:

import matplotlib.pyplot as plt

labels = ['Naive DTW', 'UCR-DTW']
values = [2.3912, 3.1907]

plt.figure(figsize=(5, 3))
plt.bar(labels, values)
plt.ylabel('Seconds (mean of 3 runs)')
plt.tight_layout()
plt.savefig('partB/results/failure_runtime.png', dpi=160)
print('Saved: partB/results/failure_runtime.png')


Saved: partB/results/failure_runtime.png


![Failure runtime](partB/results/failure_runtime.png)